In [ ]:
"""
CarbonTatva AI — Additional Models for Benchmarking
=====================================================
Model 4: Linear Regression + Ridge  (simplest baseline)
Model 5: XGBoost                    (gradient boosting alternative to LightGBM)
Model 6: N-BEATS                    (pure deep learning, no feature engineering)

Install:
    pip install scikit-learn xgboost torch

Usage:
    python extra_models.py

All outputs saved to: outputs_extra/
"""

import os
import warnings
import pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
os.makedirs('outputs_extra', exist_ok=True)

# ══════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════
DATA_FILE   = 'final_merged_data.xlsx'
MARKETS     = ['GDAM', 'DAM', 'RTM']
SLOTS_DAY   = 96
TRAIN_END   = '2024-12-31'
VAL_END     = '2025-09-30'
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ══════════════════════════════════════════════════════════════════
# STEP 1 — LOAD & FEATURE ENGINEERING (same as train_models.py)
# ══════════════════════════════════════════════════════════════════
print("=" * 60)
print("Loading and transforming data...")
print("=" * 60)

sheets = pd.read_excel(DATA_FILE, sheet_name=None)


def build_features(df):
    df = df.copy()

    # Datetime
    d = pd.to_datetime(df['Delivery Date'], dayfirst=True)
    t = df['Time Period'].str.split('-').str[0]
    df['datetime'] = pd.to_datetime(d.astype(str) + ' ' + t)
    df = df.sort_values('datetime').reset_index(drop=True)

    # Drop redundant columns
    df = df.drop(columns=[
        'apparent_temperature (°C)',
        'sunshine_duration (s)',
        'precipitation (mm)'
    ], errors='ignore')

    # Temporal
    df['hour']  = df['datetime'].dt.hour
    df['dow']   = df['datetime'].dt.dayofweek
    df['month'] = df['datetime'].dt.month

    df['hour_sin']  = np.sin(2 * np.pi * df['hour']  / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['hour']  / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['dow_sin']   = np.sin(2 * np.pi * df['dow']   / 7)
    df['dow_cos']   = np.cos(2 * np.pi * df['dow']   / 7)
    df['is_weekend'] = (df['dow'] >= 5).astype(int)

    try:
        import holidays
        india_h = holidays.India(years=range(2022, 2027))
        df['holiday_flag'] = df['datetime'].dt.date.apply(
            lambda d: 1 if d in india_h else 0)
    except ImportError:
        df['holiday_flag'] = 0

    # Autoregressive
    p = df['Price (Rs./MWh)']
    df['price_lag96']  = p.shift(96)
    df['price_lag672'] = p.shift(672)
    df['ma3d'] = p.shift(1).rolling(288).mean()
    df['ma7d'] = p.shift(1).rolling(672).mean()

    # Weather
    df['temp']     = df['temperature_2m (°C)']
    df['temp_sq']  = df['temp'] ** 2
    df['radiation'] = df['shortwave_radiation (W/m²)']
    df['radiation_x_hour_sin'] = df['radiation'] * df['hour_sin']
    df['wind_speed']  = df['wind_speed_100m (km/h)']
    df['cloud_cover'] = df['cloud_cover (%)']
    df['humidity']    = df['relative_humidity_2m (%)']
    df['rain']        = df['rain (mm)']

    df['log_price'] = np.log1p(df['Price (Rs./MWh)'])
    return df


market_dfs = {}
for mkt in MARKETS:
    print(f"  {mkt}...", end=' ')
    market_dfs[mkt] = build_features(sheets[mkt])
    print(f"{len(market_dfs[mkt]):,} rows")

# Full feature list used by tabular models
TABULAR_FEATURES = [
    'hour_sin', 'hour_cos',
    'month_sin', 'month_cos',
    'dow_sin', 'dow_cos',
    'is_weekend', 'holiday_flag',
    'price_lag96', 'price_lag672',
    'ma3d', 'ma7d',
    'radiation', 'radiation_x_hour_sin',
    'temp', 'temp_sq',
    'wind_speed', 'cloud_cover',
    'humidity', 'rain',
]

# ══════════════════════════════════════════════════════════════════
# SHARED UTILITIES
# ══════════════════════════════════════════════════════════════════
from sklearn.preprocessing import StandardScaler


def split_scale(df, feature_cols):
    df = df.dropna(subset=feature_cols + ['Price (Rs./MWh)']).copy()
    train = df[df['datetime'] <= TRAIN_END]
    val   = df[(df['datetime'] > TRAIN_END) & (df['datetime'] <= VAL_END)]
    test  = df[df['datetime'] > VAL_END]

    scaler = StandardScaler()
    tr_s, va_s, te_s = train.copy(), val.copy(), test.copy()
    tr_s[feature_cols] = scaler.fit_transform(train[feature_cols])
    va_s[feature_cols] = scaler.transform(val[feature_cols])
    te_s[feature_cols] = scaler.transform(test[feature_cols])

    print(f"    Train {len(train):,} | Val {len(val):,} | Test {len(test):,}")
    return tr_s, va_s, te_s, scaler, train, val, test


def metrics(y_true, y_pred, label=''):
    mae  = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mask = y_true > 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    print(f"    {label:25s}  MAE={mae:7.1f}  RMSE={rmse:7.1f}  MAPE={mape:.2f}%")
    return {'mae': mae, 'rmse': rmse, 'mape': mape}


def pred_plot(y_true, y_pred, title, path):
    fig, axes = plt.subplots(2, 1, figsize=(16, 8))
    fig.patch.set_facecolor('#0f1117')
    for ax in axes:
        ax.set_facecolor('#1a1d27')
        ax.tick_params(colors='#aaa')

    n = min(500, len(y_true))
    axes[0].plot(y_true[:n], color='#aaa', lw=0.8, label='Actual')
    axes[0].plot(y_pred[:n], color='#e67e22', lw=0.8, label='Predicted')
    axes[0].set_title(f'{title} — Test Set (first {n} slots)', color='white')
    axes[0].legend(fontsize=9)
    axes[0].set_ylabel('Price (Rs./MWh)', color='#aaa')

    n2 = min(SLOTS_DAY * 3, len(y_true))
    axes[1].plot(y_true[:n2], color='#aaa', lw=1.2, label='Actual')
    axes[1].plot(y_pred[:n2], color='#e67e22', lw=1.2, ls='--', label='Predicted')
    axes[1].set_title(f'{title} — First 3 Days Zoomed', color='white')
    axes[1].legend(fontsize=9)
    axes[1].set_ylabel('Price (Rs./MWh)', color='#aaa')
    axes[1].set_xlabel('Slot (15 min each)', color='#aaa')

    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight', facecolor='#0f1117')
    plt.close()


all_metrics = {}

Loading and transforming data...
  GDAM... 144,864 rows
  DAM... 144,864 rows
  RTM... 144,116 rows


In [ ]:

# ══════════════════════════════════════════════════════════════════
# MODEL 6 — N-BEATS
# ══════════════════════════════════════════════════════════════════
"""
What it is:
    Neural Basis Expansion Analysis for Time Series (N-BEATS).
    Published by Element AI (2020). It is a pure deep learning
    forecasting model that requires NO feature engineering at all —
    it only takes raw price history as input.

    N-BEATS learns to decompose a time series into interpretable
    components (trend + seasonality) using residual stacks of
    fully-connected layers. Each "block" forecasts part of the
    signal and passes the residual to the next block.

Why run it:
    - Zero feature engineering needed — just raw prices
    - Excellent benchmark: if N-BEATS (no features) matches
      LightGBM (many features), it means the autoregressive
      signal is so strong that weather/temporal features add
      little extra value
    - One of the top-performing models on the M4 and M5
      forecasting competitions
    - Naturally learns the daily/weekly seasonality from the
      raw price sequence without being told about it

Architecture used here:
    - Trend stack:       3 blocks, degree-3 polynomial basis
    - Seasonality stack: 3 blocks, Fourier basis (period=96)
    - Each block: 4 FC layers of 256 units
    - Lookback: 7 days (672 slots)
    - Horizon:  1 day  (96 slots)

Limitation:
    - Cannot directly use weather forecasts as input
    - Slower to train than LightGBM
    - Needs large dataset to work well (you have enough: ~145k rows)
"""
print("\n" + "=" * 60)
print("MODEL 6: N-BEATS")
print("=" * 60)

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    TORCH_AVAILABLE = True
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"  Device: {DEVICE}")
except ImportError:
    print("  [SKIP] pip install torch")
    TORCH_AVAILABLE = False


class NBEATSBlock(nn.Module):
    """
    Single N-BEATS block.
    Learns a backcast (what it explains from the input)
    and a forecast (what it predicts for the future).
    """
    def __init__(self, input_size, theta_size, basis_function,
                 layers=4, layer_size=256):
        super().__init__()
        self.basis_function = basis_function

        # 4 fully-connected layers with ReLU
        fc = []
        in_dim = input_size
        for _ in range(layers):
            fc += [nn.Linear(in_dim, layer_size), nn.ReLU()]
            in_dim = layer_size
        self.fc_stack = nn.Sequential(*fc)

        # Theta: coefficients for the basis functions
        self.theta_f = nn.Linear(layer_size, theta_size, bias=False)
        self.theta_b = nn.Linear(layer_size, theta_size, bias=False)

    def forward(self, x):
        h = self.fc_stack(x)
        theta_f = self.theta_f(h)
        theta_b = self.theta_b(h)
        forecast  = self.basis_function.forward(theta_f)
        backcast  = self.basis_function.backcast(theta_b)
        return backcast, forecast


class TrendBasis(nn.Module):
    """Polynomial trend basis of given degree."""
    def __init__(self, degree, backcast_size, forecast_size):
        super().__init__()
        self.forecast_size = forecast_size
        self.backcast_size = backcast_size
        # Precompute polynomial time vectors
        forecast_t = torch.arange(forecast_size).float() / forecast_size
        backcast_t = torch.arange(backcast_size).float() / backcast_size
        self.register_buffer('T_f', torch.stack(
            [forecast_t ** i for i in range(degree + 1)], dim=0))
        self.register_buffer('T_b', torch.stack(
            [backcast_t ** i for i in range(degree + 1)], dim=0))

    def forward(self, theta):
        return theta @ self.T_f

    def backcast(self, theta):
        return theta @ self.T_b


class SeasonalityBasis(nn.Module):
    """Fourier seasonality basis."""
    def __init__(self, harmonics, backcast_size, forecast_size):
        super().__init__()
        freq = torch.arange(1, harmonics + 1).float()
        forecast_t = torch.arange(forecast_size).float() / forecast_size
        backcast_t = torch.arange(backcast_size).float() / backcast_size

        # F_f shape: (2 * harmonics, forecast_size)
        F_f = torch.cat([
            torch.sin(2 * np.pi * freq[:, None] * forecast_t[None, :]),
            torch.cos(2 * np.pi * freq[:, None] * forecast_t[None, :]),
        ], dim=0)

        # F_b shape: (2 * harmonics, backcast_size)
        F_b = torch.cat([
            torch.sin(2 * np.pi * freq[:, None] * backcast_t[None, :]),
            torch.cos(2 * np.pi * freq[:, None] * backcast_t[None, :]),
        ], dim=0)

        self.register_buffer('F_f', F_f)
        self.register_buffer('F_b', F_b)

    def forward(self, theta):
        # theta is (batch, 2*harmonics) @ F_f (2*harmonics, forecast_size)
        return theta @ self.F_f

    def backcast(self, theta):
        # theta is (batch, 2*harmonics) @ F_b (2*harmonics, backcast_size)
        return theta @ self.F_b


class NBEATS(nn.Module):
    """
    Full N-BEATS model with two stacks:
      Stack 1: Trend     (polynomial basis, degree=3)
      Stack 2: Seasonality (Fourier basis, harmonics=period//2)
    """
    def __init__(self, backcast_size=SLOTS_DAY * 7,
                 forecast_size=SLOTS_DAY,
                 trend_blocks=3, season_blocks=3,
                 layers=4, layer_size=256,
                 trend_degree=3, harmonics=None):
        super().__init__()
        if harmonics is None:
            harmonics = forecast_size // 2

        # Trend stack
        trend_basis = TrendBasis(trend_degree, backcast_size, forecast_size)
        trend_theta = trend_degree + 1
        self.trend_stack = nn.ModuleList([
            NBEATSBlock(backcast_size, trend_theta, trend_basis,
                        layers, layer_size)
            for _ in range(trend_blocks)
        ])

        # Seasonality stack
        season_basis = SeasonalityBasis(harmonics, backcast_size, forecast_size)
        season_theta = harmonics * 2
        self.season_stack = nn.ModuleList([
            NBEATSBlock(backcast_size, season_theta, season_basis,
                        layers, layer_size)
            for _ in range(season_blocks)
        ])

    def forward(self, x):
        residual = x
        forecast = torch.zeros(x.shape[0], SLOTS_DAY).to(x.device)

        for block in self.trend_stack:
            backcast, f = block(residual)
            residual = residual - backcast
            forecast = forecast + f

        for block in self.season_stack:
            backcast, f = block(residual)
            residual = residual - backcast
            forecast = forecast + f

        return forecast


class NBEATSDataset(Dataset):
    """Raw price sequence — no features, just prices."""
    def __init__(self, prices, lookback=SLOTS_DAY * 7, horizon=SLOTS_DAY):
        self.prices  = prices.astype(np.float32)
        self.lookback = lookback
        self.horizon  = horizon

    def __len__(self):
        return len(self.prices) - self.lookback - self.horizon + 1

    def __getitem__(self, idx):
        x = self.prices[idx: idx + self.lookback]
        y = self.prices[idx + self.lookback: idx + self.lookback + self.horizon]
        return torch.tensor(x), torch.tensor(y)


def train_nbeats(mkt, df, lookback=SLOTS_DAY * 7, horizon=SLOTS_DAY,
                 epochs=60, batch_size=32, lr=0.001):
    print(f"\n  Training N-BEATS for {mkt}...")

    # N-BEATS only needs prices — normalise them
    df_clean = df.dropna(subset=['Price (Rs./MWh)']).copy()
    prices   = df_clean['Price (Rs./MWh)'].values

    # Normalise: log + z-score
    log_prices = np.log1p(prices)
    mu, sigma  = log_prices.mean(), log_prices.std()
    norm_prices = (log_prices - mu) / sigma

    # Time-based split by index proportion
    n = len(norm_prices)
    n_train = int(n * 0.75)
    n_val   = int(n * 0.87)

    train_prices = norm_prices[:n_train]
    val_prices   = norm_prices[n_train - lookback: n_val]
    test_prices  = norm_prices[n_val   - lookback:]

    print(f"    Train {len(train_prices):,} | "
          f"Val {len(val_prices):,} | Test {len(test_prices):,} slots")

    train_ds = NBEATSDataset(train_prices, lookback, horizon)
    val_ds   = NBEATSDataset(val_prices,   lookback, horizon)
    test_ds  = NBEATSDataset(test_prices,  lookback, horizon)

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size,
                              shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size,
                              shuffle=False, num_workers=0)

    model = NBEATS(
        backcast_size=lookback,
        forecast_size=horizon,
        trend_blocks=3,
        season_blocks=3,
        layers=4,
        layer_size=256,
        trend_degree=3,
        harmonics=horizon // 2,
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs)
    criterion = nn.HuberLoss(delta=1.0)

    best_val = float('inf')
    patience_counter = 0
    PATIENCE = 10
    train_losses, val_losses = [], []

    for epoch in range(1, epochs + 1):
        model.train()
        ep_loss = 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            pred = model(x)
            loss = criterion(pred, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ep_loss += loss.item()

        model.eval()
        vl = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                vl += criterion(model(x), y).item()

        train_losses.append(ep_loss / len(train_loader))
        val_losses.append(vl / len(val_loader))
        scheduler.step()

        if epoch % 10 == 0:
            print(f"    Epoch {epoch:3d}/{epochs}  "
                  f"train={train_losses[-1]:.4f}  "
                  f"val={val_losses[-1]:.4f}")

        if val_losses[-1] < best_val:
            best_val = val_losses[-1]
            patience_counter = 0
            torch.save(model.state_dict(),
                       f'outputs_extra/nbeats_{mkt}_best.pt')
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"    Early stopping at epoch {epoch}")
                break

    # Load best
    model.load_state_dict(torch.load(
        f'outputs_extra/nbeats_{mkt}_best.pt', map_location=DEVICE))

    # Evaluate
    model.eval()
    preds_norm, targets_norm = [], []
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(DEVICE)
            p = model(x).cpu().numpy()
            preds_norm.append(p)
            targets_norm.append(y.numpy())

    preds_norm  = np.concatenate(preds_norm).flatten()
    target_norm = np.concatenate(targets_norm).flatten()

    # Inverse normalise: z-score inverse → log inverse
    preds_log   = preds_norm * sigma + mu
    targets_log = target_norm * sigma + mu
    preds_price  = np.clip(np.expm1(preds_log),  0, 20000)
    target_price = np.clip(np.expm1(targets_log), 0, 20000)

    print(f"  {mkt} results:")
    m = metrics(target_price, preds_price, 'Test')
    all_metrics['nbeats'][mkt] = m

    # Training curve
    fig, ax = plt.subplots(figsize=(10, 4))
    fig.patch.set_facecolor('#0f1117')
    ax.set_facecolor('#1a1d27')
    ax.plot(train_losses, color='#e74c3c', label='Train Loss')
    ax.plot(val_losses,   color='#27ae60', label='Val Loss')
    ax.set_title(f'N-BEATS Training Curve — {mkt}', color='white')
    ax.set_xlabel('Epoch', color='#aaa')
    ax.set_ylabel('Huber Loss', color='#aaa')
    ax.legend(fontsize=9)
    ax.tick_params(colors='#aaa')
    plt.tight_layout()
    plt.savefig(f'outputs_extra/nbeats_loss_{mkt}.png',
                dpi=130, bbox_inches='tight', facecolor='#0f1117')
    plt.close()

    pred_plot(target_price, preds_price,
              f'N-BEATS {mkt}',
              f'outputs_extra/nbeats_pred_{mkt}.png')

    print(f"  Saved: outputs_extra/nbeats_{mkt}_best.pt")
    return model, mu, sigma


if TORCH_AVAILABLE:
    all_metrics['nbeats'] = {}
    nbeats_models = {}
    for mkt in MARKETS:
        nbeats_models[mkt] = train_nbeats(mkt, market_dfs[mkt])



MODEL 6: N-BEATS
  Device: cuda

  Training N-BEATS for GDAM...
    Train 108,648 | Val 18,055 | Test 19,505 slots
    Epoch  10/60  train=0.0313  val=0.1337
    Early stopping at epoch 11
  GDAM results:
    Test                       MAE=  590.0  RMSE=  953.9  MAPE=16.23%
  Saved: outputs_extra/nbeats_GDAM_best.pt

  Training N-BEATS for DAM...
    Train 108,648 | Val 18,055 | Test 19,505 slots
    Epoch  10/60  train=0.0225  val=0.1254
    Early stopping at epoch 12
  DAM results:
    Test                       MAE=  624.5  RMSE= 1039.7  MAPE=32.72%
  Saved: outputs_extra/nbeats_DAM_best.pt

  Training N-BEATS for RTM...
    Train 108,087 | Val 17,965 | Test 19,408 slots
    Epoch  10/60  train=0.0518  val=0.2229
    Early stopping at epoch 11
  RTM results:
    Test                       MAE=  885.4  RMSE= 1386.2  MAPE=544.86%
  Saved: outputs_extra/nbeats_RTM_best.pt


In [ ]:
# ══════════════════════════════════════════════════════════════════
# FINAL COMPARISON — All 6 Models
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("FULL MODEL COMPARISON (Extra Models)")
print("=" * 60)

model_labels = {
    'nbeats':  'N-BEATS',
}

rows = []
for model_key, model_name in model_labels.items():
    if model_key not in all_metrics:
        continue
    for mkt, result in all_metrics[model_key].items():
        m = result.get('test', result)
        rows.append({
            'Model':  model_name,
            'Market': mkt,
            'MAE':    round(m['mae'],  1),
            'RMSE':   round(m['rmse'], 1),
            'MAPE%':  round(m['mape'], 2),
        })

if rows:
    df_summary = pd.DataFrame(rows)
    print(df_summary.to_string(index=False))
    df_summary.to_csv('outputs_extra/extra_models_metrics.csv', index=False)
    print("\nSaved: outputs_extra/extra_models_metrics.csv")

    # Bar chart comparison
    model_colors = {
        'Ridge Regression': '#8e44ad',
        'XGBoost':          '#e67e22',
        'N-BEATS':          '#16a085',
    }
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.patch.set_facecolor('#0f1117')
    fig.suptitle('Extra Models — MAE Comparison by Market',
                 color='white', fontsize=14)

    for ax, mkt in zip(axes, MARKETS):
        ax.set_facecolor('#1a1d27')
        sub = df_summary[df_summary['Market'] == mkt]
        bars = ax.bar(
            sub['Model'], sub['MAE'],
            color=[model_colors.get(m, '#aaa') for m in sub['Model']],
            alpha=0.85
        )
        ax.set_title(mkt, color='white')
        ax.set_ylabel('MAE (Rs./MWh)', color='#aaa')
        ax.tick_params(colors='#aaa', axis='both')
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=15,
                 ha='right', fontsize=8)
        for bar, val in zip(bars, sub['MAE']):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 5, f'{val:.0f}',
                    ha='center', color='white', fontsize=9)

    plt.tight_layout()
    plt.savefig('outputs_extra/extra_models_comparison.png',
                dpi=130, bbox_inches='tight', facecolor='#0f1117')
    plt.close()
    print("Saved: outputs_extra/extra_models_comparison.png")

print("\n✅ Extra models done! Check outputs_extra/")
print("   Key files:")
print("   outputs_extra/ridge_{MARKET}.pkl        — Ridge model + scaler")
print("   outputs_extra/xgb_{MARKET}.json         — XGBoost model")
print("   outputs_extra/nbeats_{MARKET}_best.pt   — N-BEATS weights")
print("   outputs_extra/extra_models_metrics.csv  — All metrics")
print("   outputs_extra/extra_models_comparison.png")



FULL MODEL COMPARISON (Extra Models)
  Model Market   MAE   RMSE  MAPE%
N-BEATS   GDAM 590.0  953.9  16.23
N-BEATS    DAM 624.5 1039.7  32.72
N-BEATS    RTM 885.4 1386.2 544.86

Saved: outputs_extra/extra_models_metrics.csv
Saved: outputs_extra/extra_models_comparison.png

✅ Extra models done! Check outputs_extra/
   Key files:
   outputs_extra/ridge_{MARKET}.pkl        — Ridge model + scaler
   outputs_extra/xgb_{MARKET}.json         — XGBoost model
   outputs_extra/nbeats_{MARKET}_best.pt   — N-BEATS weights
   outputs_extra/extra_models_metrics.csv  — All metrics
   outputs_extra/extra_models_comparison.png


In [ ]:
import os
from google.colab import files

# 1. Zip the 'outputs' folder
# The syntax is: !zip -r [target_zip_name] [source_folder_path]
!zip -r /content/outputs_extra.zip /content/outputs_extra


  adding: content/outputs_extra/ (stored 0%)
  adding: content/outputs_extra/nbeats_loss_RTM.png (deflated 11%)
  adding: content/outputs_extra/nbeats_DAM_best.pt (deflated 8%)
  adding: content/outputs_extra/nbeats_loss_GDAM.png (deflated 10%)
  adding: content/outputs_extra/extra_models_comparison.png (deflated 17%)
  adding: content/outputs_extra/nbeats_pred_DAM.png (deflated 4%)
  adding: content/outputs_extra/nbeats_pred_GDAM.png (deflated 4%)
  adding: content/outputs_extra/nbeats_pred_RTM.png (deflated 3%)
  adding: content/outputs_extra/nbeats_loss_DAM.png (deflated 11%)
  adding: content/outputs_extra/nbeats_RTM_best.pt (deflated 8%)
  adding: content/outputs_extra/nbeats_GDAM_best.pt (deflated 8%)
  adding: content/outputs_extra/extra_models_metrics.csv (deflated 16%)


In [ ]:
# 2. Download the file to your local machine
files.download('/content/outputs_extra.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>